# RHAPSODY Ensemble Backend Tutorial

This tutorial shows how to use the **`EnsembleExecutionBackend`** to run tasks on EnsembleLauncher cluster:

In [1]:
import asyncio

from rhapsody.api import ComputeTask, Session
from rhapsody.backends import EnsembleBackend

---

## 1. Execute Functions

In [2]:
def compute_square(n):
    """Compute the square of a number."""
    return {"input": n, "result": n * n}

In [3]:
async def run_sync_functions():
    async with EnsembleBackend() as backend:
        session = Session(backends=[backend])
        tasks = [ComputeTask(function=compute_square, args=(i,)) for i in range(20)]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        done = sum(1 for t in tasks if t.state == "DONE")
        failed = sum(1 for t in tasks if t.state == "FAILED")
        print(f"Done: {done}, Failed: {failed}")
        for t in tasks[:3]:
            print(f"  {t.uid} -> {t.return_value}")

await run_sync_functions()

Done: 20, Failed: 0
  task.000001 -> {'input': 0, 'result': 0}
  task.000002 -> {'input': 1, 'result': 1}
  task.000003 -> {'input': 2, 'result': 4}


**Expected output:**
```
Done: 20, Failed: 0
  task.000001 -> {'input': 0, 'result': 0}
  task.000002 -> {'input': 1, 'result': 1}
  task.000003 -> {'input': 2, 'result': 4}
```

---

## 2. Executable Tasks

Executable task results are available via `task.stdout`, `task.stderr`.

In [4]:
async def run_executables():
    async with EnsembleBackend() as backend:
        session = Session(backends=[backend])
        tasks = [
            ComputeTask(executable="/bin/echo", arguments=[f"task {i}"])
            for i in range(5)
        ]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        for t in tasks:
            print(f"{t.uid} | exit={t.exit_code} | stdout={t.stdout.strip()!r}")

await run_executables()

task.000021 | exit=None | stdout='task 0'
task.000022 | exit=None | stdout='task 1'
task.000023 | exit=None | stdout='task 2'
task.000024 | exit=None | stdout='task 3'
task.000025 | exit=None | stdout='task 4'


**Expected output:**
```
task.000001 | exit=0 | stdout='task 0'
task.000002 | exit=0 | stdout='task 1'
...
```

## 3. Resource awareness and pinning
EnsembleBackend supports resource-aware scheduling and pinning of tasks to specific resources. gpu pinning is controlled through `gpu_selector` argument of `EnsembleExecutionBackend`. The following example shows how to pin tasks to specific GPUs. The below example just shows that each task will have specific `gpu_selector` set correctly in its env. Note, `cpu_affinity` is only ensured when python `os` module allows it. Consequently, the `cpu_affinity` may not be set on some platforms, like macos.

In [5]:
async def run_executables():
    async with EnsembleBackend(gpu_selector="ZE_AFFINITY_MASK", gpus=[0, 1]) as backend:
        session = Session(backends=[backend])
        tasks = [
            ComputeTask(executable="printenv", arguments=["ZE_AFFINITY_MASK"], task_backend_specific_kwargs = {"ranks": 1, "gpus_per_rank": 1, "cpu_affinity":f"{i}", "gpu_affinity": f"{gpu_id}"})
            for i, gpu_id in zip(list(range(2)),[1, 0]) #notice that the affinity is set opposite to the i
        ]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        for t in tasks:
            print(f"{t.uid} | exit={t.exit_code} | stdout={t.stdout.strip()!r}")

await run_executables()

task.000026 | exit=None | stdout='1'
task.000027 | exit=None | stdout='0'


## 4. Heteregeneous tasks
The example shows how to execute MPI + CPU + GPU tasks.

In [6]:
def echo_hello():
    return "Hello CPU"

async def run_executables():
    async with EnsembleBackend(mpi_flavour="test") as backend:
        session = Session(backends=[backend])
        mpi_task = ComputeTask(executable="printenv", arguments=["OMPI_COMM_WORLD_LOCAL_RANK"], task_backend_specific_kwargs = {"ranks": 2})
        serial_task = ComputeTask(function=echo_hello)
        tasks = [mpi_task, serial_task]
        async with session:
            await session.submit_tasks(tasks)
            await session.wait_tasks(tasks)

        for t in tasks:
            print(f"{t.uid} | stdout={t.stdout.strip()!r} | return={t.return_value}")

await run_executables()

task.000028 | stdout='0\n1' | return=
task.000029 | stdout='' | return=Hello CPU
